# TatHybrid Dataset Experiment

**Dataset:** TatHybrid (Financial Reports - Numeracy Focused)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Numeracy F1 (numeracy-aware metric)

**Documents:** 4 example PDFs
- inpixon_2019.pdf
- lifeway-foods-inc_2019.pdf
- overseas-shipholding-group-inc_2019.pdf
- viavi-solutions-inc_2019.pdf

**Parameters:**
- Chunk size: 3000 characters
- Chunk overlap: 300 characters
- Top-K retrieval: 5
- Temperature: 0.1
- Max tokens: 512

## Setup and Imports

In [1]:
import sys
import os

# CRITICAL: Change to project root directory
# The preprocess module uses relative paths from project root
project_root = os.path.abspath('../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess, llm
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ All imports successful


## Configuration

In [2]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [3]:
# Experiment Parameters
DATASET_NAME = "tat"
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300
TOP_K = 5
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/1_without_optimization/tathybrid/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: tat
Chunk size: 3000
Top-K: 5
Output dir: ./experiments/tathybrid/results


## Initialize Models

In [4]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model (local, free)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

✓ Together AI client initialized


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized


## Helper Functions

In [5]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """Retrieve context and generate answer"""
    # Retrieve
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # Build prompt
    llm_message = llm.make_prompt(
        question=question,
        context=context,
        task_name=DATASET_NAME,
        llm_type="gpt-4"
    )
    
    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=llm_message,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

✓ Helper functions defined


## Load Q&A Data

In [6]:
# Load TatHybrid Q&A
csv_file = "./dataset/qa/tat_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# IMPORTANT: Filter to only documents with available PDFs in example directory
# Only these 4 PDFs are in dataset/src_doc_files_example/tat_docs/
AVAILABLE_DOCS = [
    "inpixon_2019",
    "lifeway-foods-inc_2019",
    "overseas-shipholding-group-inc_2019",
    "viavi-solutions-inc_2019"
]

# Filter Q&A dict to only available documents
qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

# Count Q&A per available document
total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")

Total documents in CSV: 170
Available PDFs: 4

Filtered to documents with PDFs:

  inpixon_2019: 18 Q&A pairs
  lifeway-foods-inc_2019: 60 Q&A pairs
  overseas-shipholding-group-inc_2019: 60 Q&A pairs
  viavi-solutions-inc_2019: 24 Q&A pairs

Total Q&A to process: 162


## Main Processing Loop

**This will process 4 documents with their Q&A pairs**

**Expected runtime:** 30-60 minutes (depends on total Q&A count)

In [7]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index
    print("Building vector index...")
    collection = build_index(text_chunks, collection_name=f"tat_{doc_name}")
    print("✓ Index built")
    
    # Process each question
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")


Processing: overseas-shipholding-group-inc_2019
PDF: dataset/src_doc_files_example/tat_docs/overseas-shipholding-group-inc_2019.pdf
Extracting text...
Created 143 chunks
Building vector index...
✓ Index built

Answering 60 questions...

[1/60] What benefits are provided by the company to qualifying domestic retir...
   Answer: The answer is: Postretirement health care and life insurance benefits...

[2/60] What is the change in Interest cost on benefit obligation for pension ...
   Answer: The answer is: 129...

[3/60] What is the average Interest cost on benefit obligation for pension be...
   Answer: The answer is: 1,737.5...

[4/60] In which year was Benefit obligation at beginning of year for pension ...
   Answer: ...

[5/60] What was the Interest cost on benefit obligation in 2019 and 2018 resp...
   Answer: ...

[6/60] What was the Benefit obligation at year end of 2019 for pension benefi...
   Answer: The answer is: $46,936 thousand...

[7/60] When did the Company sold one ATB

## Diagnostic: Check Empty Responses

if all_results:
    import pandas as pd
    
    # Create DataFrame for analysis
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)
    
    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")
    
    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
        
        print(f"\nSample empty questions (first 5):")
        empty_df = results_df[results_df['is_empty']].head(5)
        for idx, row in empty_df.iterrows():
            print(f"\n  [{idx+1}] {row['question'][:80]}...")
            print(f"      Doc: {row['doc']}")
            print(f"      Response: '{row['response']}'")
    else:
        print("\n✓ All questions received answers!")
    
    print(f"\nSample answered questions (first 3):")
    answered_df = results_df[~results_df['is_empty']].head(3)
    for idx, row in answered_df.iterrows():
        print(f"\n  Q: {row['question'][:80]}...")
        print(f"  A: {row['response'][:100]}...")

In [9]:
if all_results:
    print("\nEvaluating TatHybrid results (Numeracy F1)...")
    
    # WORKAROUND: Fix answer format for TatQA evaluation
    # The eval code expects answer['answer'] to be a single value for count type,
    # but it's stored as a list in the CSV
    fixed_results = []
    for result in all_results:
        fixed_result = result.copy()
        answers = fixed_result['answers']
        if isinstance(answers, dict) and 'answer' in answers:
            # If answer is a list with one element, extract it for count/arithmetic types
            if isinstance(answers['answer'], list) and len(answers['answer']) == 1:
                if answers.get('answer_type') in ['count', 'arithmetic']:
                    # Keep the dict structure but fix the answer field
                    fixed_result['answers'] = {
                        'answer': answers['answer'][0],  # Extract single value
                        'answer_type': answers['answer_type'],
                        'scale': answers['scale']
                    }
        fixed_results.append(fixed_result)
    
    eval_main(DATASET_NAME, fixed_results)
else:
    print("❌ No results to evaluate")


Evaluating TatHybrid results (Numeracy F1)...
Numerical F1 score: 43.50


## Save Results

In [10]:
if all_results:
    # Save to CSV
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"tathybrid_results_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/tathybrid/results/tathybrid_results_20260629_094232.csv
Total Q&A: 162

Results by document:
  overseas-shipholding-group-inc_2019: 60 questions
  lifeway-foods-inc_2019: 60 questions
  viavi-solutions-inc_2019: 24 questions
  inpixon_2019: 18 questions


## Summary Statistics

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    empty_count = results_df['response'].str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    print(f"\n{'='*80}")
    print(f"STATISTICS")
    print(f"{'='*80}")
    print(f"Total questions: {len(results_df)}")
    print(f"Answered: {answered_count} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"Empty responses: {empty_count} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"Avg response length: {results_df['response'].str.len().mean():.0f} characters")


STATISTICS
Total questions: 162
Answered: 125 (77.2%)
Empty responses: 37 (22.8%)
Avg response length: 66 characters


---

## Done!

**Results saved to:** `./results/tathybrid_results_[timestamp].csv`

**Metric:** Numeracy F1 (numeracy-aware evaluation)

**Next steps:**
1. Review the Numeracy F1 score
2. Compare with FinHybrid results
3. Analyze numeracy-specific performance